In [2]:
import json
import pandas as pd


In [4]:
bairro = 'Itaim Bibi'
codigo = '00005'

In [5]:
df = pd.read_json(f'dados/{codigo}/{bairro} - proprietarios_anuncios.json')

### Limpeza dos dados

In [6]:
dados_proprietarios = pd.json_normalize(df['data'])
dados_proprietarios = dados_proprietarios.drop([
    'searchId',
    'totalPages',
], axis = 1)

In [7]:
df = df.drop([
    'notification',
    'message',
    'error',
    'notification_alert',
    'data',
], axis = 1)

In [8]:
filtered_df = pd.concat([df, dados_proprietarios], axis = 1)

Filtrando para somente imóveis com 1 possível proprietário

In [9]:
filtered_df = filtered_df[filtered_df['totalElements'] == 1]

In [10]:
filtered_df = filtered_df.reset_index(drop=True)

In [11]:
filtered_df.head()

,announce_id,totalElements,people
0,293616247,1,"[{'document': '18160464000120', 'name': 'Upcon..."
1,293616165,1,"[{'document': '80922325804', 'name': 'Monica T..."
2,293541774,1,"[{'document': '02150825830', 'name': 'Albert M..."
3,293616247,1,"[{'document': '18160464000120', 'name': 'Upcon..."
4,293616248,1,"[{'document': '18160464000120', 'name': 'Upcon..."


### Extraindo informações dos proprietários

In [12]:
contact_df = pd.json_normalize(filtered_df['people'].apply(lambda x: x[0]))

In [13]:
contact_df.head()

,document,name,birthDate,type,addresses
0,18160464000120,Upcon Spe 26 Empreendimentos Imobiliarios S A,,PESSOA_FISICA,"[{'street': 'RUA BANDEIRA PAULISTA', 'number':..."
1,80922325804,Monica Taubkin,,PESSOA_FISICA,"[{'street': 'RUA LOPES NETO', 'number': 80, 'e..."
2,02150825830,Albert Mizrahi,,PESSOA_FISICA,"[{'street': 'AVENIDA HORACIO LAFER', 'number':..."
3,18160464000120,Upcon Spe 26 Empreendimentos Imobiliarios S A,,PESSOA_FISICA,"[{'street': 'RUA BANDEIRA PAULISTA', 'number':..."
4,18160464000120,Upcon Spe 26 Empreendimentos Imobiliarios S A,,PESSOA_FISICA,"[{'street': 'RUA BANDEIRA PAULISTA', 'number':..."


In [14]:
contact_df = contact_df[['document', 'name']]

In [15]:
filtered_df = pd.concat([filtered_df, contact_df], axis = 1, join='inner')

In [16]:
filtered_df = filtered_df[['announce_id', 'name', 'document']]

Filtrando somente CPF

In [17]:
filtered_df = filtered_df[filtered_df['document'].str.len() == 11]

In [18]:
len(filtered_df)

237

In [19]:
filtered_df.head()

,announce_id,name,document
1,293616165,Monica Taubkin,80922325804
2,293541774,Albert Mizrahi,02150825830
5,293616165,Monica Taubkin,80922325804
6,293541774,Albert Mizrahi,02150825830
7,293419906,Albert Mizrahi,02150825830


### Agrupando por CPF para análise

In [20]:
grouped_df = filtered_df.groupby('document')

aggregated_df = grouped_df.count()

aggregated_df.head()

,announce_id,name
document,,
00433622067,1,1
01257006878,1,1
01935591967,6,6
02150825830,160,160
03020214858,3,3


In [21]:
len(aggregated_df)

22

### Salvando df com valores filtrados com nossas regras

In [37]:
filtered_df.to_csv(f'dados/{bairro} - tratado.csv')